# Step 7 — Apply offline inclusion / exclusion criteria

> Filter on every IC/EC that does **not** require a live GitHub API call. The remaining online criteria are enforced in Step 8.

**Paper section:** Step 1 — Repository selection and workflow extraction (offline IC/EC enforcement); see Table 'Inclusion and exclusion criteria'.

## Config setup (read first)

This is the **first step that reads `data/mining_config.json`**. If you cloned the repo, the real config file is git-ignored and must be created from the example file before running this notebook.
On Linux you can do:
```bash
# from repo root
cp data/mining_config.example.json data/mining_config.json
```
In Windows or others you may just copy with your "File Explorer" and rename it to `mining_config.json`.
Then open `data/mining_config.json` and set at least:

- `filters.reference_date` — fixed ISO date (`YYYY-MM-DD`) of the mining snapshot (required for reproducible IC3/IC4 cutoffs in this notebook).
- `github.tokens` (or export `GITHUB_TOKEN`) — needed by Step 8 for the online enrichment API calls.

Skip this and the notebook will raise `ValueError: filters.reference_date missing from mining_config.json`.

## Inputs
| Path | Format | Source |
|---|---|---|
| `data/6_dataset.csv` | CSV | Step 6 |
| `data/mining_config.json` | JSON | Manual — copy from `data/mining_config.example.json` (see setup above) |

## Outputs
| Path | Format | Consumer |
|---|---|---|
| `data/7_dataset_filtered.csv` | CSV (post offline IC/EC) | Step 8 (online enrichment) |
| `data/step7_report.json` | JSON | Step 8 |


## Notes
- EC1 (fork) and EC2 (archived) require live API metadata (enforced in Step 8).
- IC7 (trigger) and IC8 (≥100 runs) also need workflow content / API (Step 8).


In [1]:
# --- Notebook bootstrap ---
# Idempotent: chdir to repo root + reset sys.argv for argparse.
# Works in Jupyter/JupyterLab/Antigravity/VS Code/plain python on Linux+macOS+Windows.
from dataset_utils.notebook_bootstrap import setup

setup()

print("Step 7 — starting: apply offline IC/EC criteria...", flush=True)
"""Step 7: apply offline inclusion/exclusion criteria.

Filters the dataset using criteria that do not require GitHub API access:
- IC1: repo_stars >= 100
- IC2: repo_commits >= 100
- IC3: repo_updated within last 12 months
- IC4: repository age >= 6 months
- IC5: repo_branch in ('main', 'master')
- IC6: repository has >= 2 distinct workflow files
- EC3: repository name contains exclude keywords
- EC5: workflow without modification history (single entry, type 'A') -- evaluated per-repository; fully subsumed by IC6, so deterministically 0
- EC6: workflow filename/path contains documentation-related or
       disabled/temp/backup terms
- EC7: workflow file is not under the standard .github/workflows/ path
"""

import argparse
import json
import re
from dataclasses import dataclass
from datetime import datetime, timedelta
from pathlib import Path

import polars as pl

EXCLUDE_KEYWORDS = {
    "learning",
    "tutorial",
    "clone",
    "template",
    "actions",
    "workflows",
    "reusable",
}

DOCS_KEYWORDS = {
    "docs",
    "documentation",
    "readme",
    "wiki",
    "gh-pages",
    "deploy-docs",
    "publish-docs",
    # Disabled / backup / temp workflows that are not active CI/CD
    "disabled",
    "inactive",
    "_old",
    ".old",
    "bak",
    "temp",
    "backup",
}

# Workflow files must live directly under .github/workflows/ (or .github/workflow/).
# Paths in subdirectories, missing the dot, or nested repos are not valid GHA workflows.
VALID_WORKFLOW_PATH_PREFIXES = (
    ".github/workflows/",
)

@dataclass
class FilterStats:
    total_rows: int = 0
    total_repos: int = 0
    ic1_stars: int = 0
    ic2_commits: int = 0
    ic3_active: int = 0
    ic4_age: int = 0
    ic5_branch: int = 0
    ic6_workflows: int = 0
    ec3_keywords: int = 0
    ec5_no_history: int = 0
    ec6_docs_files: int = 0
    ec7_invalid_path: int = 0
    final_rows: int = 0
    final_repos: int = 0

def load_config() -> dict:
    config_path = Path("data/mining_config.json")
    if not config_path.exists():
        raise FileNotFoundError(
            f"{config_path} not found. Reproducer setup: copy "
            "data/mining_config.example.json -> data/mining_config.json "
            "and set filters.reference_date (see notebook intro)."
        )
    with config_path.open("r") as f:
        return json.load(f)

def _load_filter_config() -> dict:
    config = load_config()
    filters = config.get("filters", {})

    return {
        "min_stars": filters.get("min_stars", 100),
        "min_commits": filters.get("min_repository_commits", 100),
        "max_last_commit_age_days": filters.get("max_last_commit_age_days", 365),
        "min_age_days": filters.get("min_repository_age_days", 183),
        "min_workflows": filters.get("min_workflow_files", 2),
        "reference_date": filters.get("reference_date"),
        "exclude_keywords": set(filters.get("exclude_keywords", list(EXCLUDE_KEYWORDS)))
        | set(filters.get("workflow_only_keywords", [])),
    }

def _resolve_reference_date(config: dict) -> datetime:
    ref = config.get("reference_date")
    if not ref:
        msg = (
            "filters.reference_date missing from mining_config.json — required "
            "for reproducible IC3/IC4 cutoffs. Set it to a fixed ISO date "
            "(YYYY-MM-DD) that represents the mining snapshot."
        )
        raise ValueError(msg)
    return datetime.strptime(ref, "%Y-%m-%d")

def apply_offline_criteria(input_path: Path, output_path: Path) -> FilterStats:
    stats = FilterStats()
    config = _load_filter_config()

    reference_date = _resolve_reference_date(config)
    cutoff_active = reference_date - timedelta(days=config["max_last_commit_age_days"])
    cutoff_age = reference_date - timedelta(days=config["min_age_days"])

    from dataset_utils.integrity import assert_input_matches_report

    input_sha = assert_input_matches_report(
        input_path=input_path,
        prior_report_path=Path("data/step6_report.json"),
        label="step 7 input",
    )

    print("  reading input CSV (may take a moment for large files)...", flush=True)
    df = pl.read_csv(input_path, encoding="utf-8-sig")
    stats.total_rows = len(df)

    repo_groups = df.group_by("repository").agg(
        [
            pl.col("file_path").n_unique().alias("unique_workflows"),
            pl.col("repo_stars").first().alias("stars"),
            pl.col("repo_commits").first().alias("commits"),
            pl.col("repo_created").first().alias("created"),
            pl.col("repo_updated").first().alias("updated"),
            pl.col("repo_branch").first().alias("branch"),
            pl.len().alias("total_rows"),
        ]
    )
    stats.total_repos = len(repo_groups)

    exclude_repos = set()

    repo_groups = repo_groups.with_columns(
        pl.col("stars").cast(pl.Int64, strict=False).alias("stars"),
        pl.col("commits").cast(pl.Int64, strict=False).alias("commits"),
        pl.col("created").str.to_date("%Y-%m-%d", strict=False).alias("created"),
        pl.col("updated").str.to_date("%Y-%m-%d", strict=False).alias("updated"),
    )

    ic1_mask = repo_groups["stars"] < config["min_stars"]
    stats.ic1_stars = ic1_mask.sum()
    exclude_repos.update(repo_groups.filter(ic1_mask)["repository"].to_list())

    ic2_mask = repo_groups["commits"] < config["min_commits"]
    stats.ic2_commits = ic2_mask.sum()
    exclude_repos.update(repo_groups.filter(ic2_mask)["repository"].to_list())

    ic3_mask = repo_groups["updated"] < cutoff_active
    stats.ic3_active = ic3_mask.sum()
    exclude_repos.update(repo_groups.filter(ic3_mask)["repository"].to_list())

    ic4_mask = repo_groups["created"] > cutoff_age
    stats.ic4_age = ic4_mask.sum()
    exclude_repos.update(repo_groups.filter(ic4_mask)["repository"].to_list())

    ic5_mask = ~repo_groups["branch"].is_in(["main", "master"])
    stats.ic5_branch = ic5_mask.sum()
    exclude_repos.update(repo_groups.filter(ic5_mask)["repository"].to_list())

    ic6_mask = repo_groups["unique_workflows"] < config["min_workflows"]
    stats.ic6_workflows = ic6_mask.sum()
    exclude_repos.update(repo_groups.filter(ic6_mask)["repository"].to_list())

    ec3_pattern = "|".join(re.escape(kw.lower()) for kw in sorted(config["exclude_keywords"]))
    ec3_repos = df.filter(
        pl.col("repository").str.to_lowercase().str.contains(ec3_pattern, literal=False)
    )["repository"].unique()
    stats.ec3_keywords = len(ec3_repos)
    exclude_repos.update(ec3_repos.to_list())

    filtered_df = df.filter(~pl.col("repository").is_in(list(exclude_repos)))

    single_entry_repos = filtered_df.group_by("repository").agg(
        pl.len().alias("count"), pl.col("git_change_type").first().alias("first_change")
    )
    ec5_mask = (single_entry_repos["count"] == 1) & (single_entry_repos["first_change"] == "A")
    stats.ec5_no_history = ec5_mask.sum()
    ec5_repos = single_entry_repos.filter(ec5_mask)["repository"].to_list()
    exclude_repos.update(ec5_repos)

    filtered_df = filtered_df.filter(~pl.col("repository").is_in(ec5_repos))

    # EC6a: filename/path contains documentation or disabled/temp/backup keywords
    # Short tokens (temp, bak) are substrings of legitimate names
    # (template, bake/buildx-bake), so require non-alnum boundaries for them;
    # the longer keywords stay plain substrings.
    DOCS_SHORT = {"bak", "temp"}
    docs_long = "|".join(re.escape(kw) for kw in sorted(DOCS_KEYWORDS - DOCS_SHORT))
    docs_short = r"(^|[^a-z0-9])(temp|bak)([^a-z0-9]|$)"
    docs_pattern = f"({docs_long})|{docs_short}"
    ec6_docs_mask = filtered_df["file_path"].str.to_lowercase().str.contains(
        docs_pattern, literal=False
    )
    stats.ec6_docs_files = ec6_docs_mask.sum()
    filtered_df = filtered_df.filter(~ec6_docs_mask)

    # EC7: path must start with .github/workflows/
    # (catches paths in wrong directories, missing dot, nested repos, etc.)
    paths_lower = filtered_df["file_path"].str.to_lowercase()
    # GHA only runs workflows DIRECTLY under the prefix dir; nested
    # subdirectories never execute. Require exactly one segment after it.
    valid_patterns = ["^" + re.escape(p) + r"[^/]+$" for p in VALID_WORKFLOW_PATH_PREFIXES]
    valid_path_mask = paths_lower.str.contains(valid_patterns[0], literal=False)
    for pat in valid_patterns[1:]:
        valid_path_mask = valid_path_mask | paths_lower.str.contains(pat, literal=False)
    ec7_invalid_path_mask = ~valid_path_mask
    stats.ec7_invalid_path = int(ec7_invalid_path_mask.sum())
    filtered_df = filtered_df.filter(~ec7_invalid_path_mask)

    final_df = filtered_df
    stats.final_rows = len(final_df)

    unique_repos_in_final = final_df["repository"].unique()
    stats.final_repos = len(unique_repos_in_final)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    print("  writing output...", flush=True)
    final_df.write_csv(output_path, include_header=True)

    report_path = output_path.parent / "step7_report.json"
    report = {
        "reference_date": reference_date.strftime("%Y-%m-%d"),
        "total_rows": stats.total_rows,
        "total_repos": stats.total_repos,
        "final_rows": stats.final_rows,
        "final_repos": stats.final_repos,
        "excluded_by_criteria": {
            "IC1_stars": stats.ic1_stars,
            "IC2_commits": stats.ic2_commits,
            "IC3_active": stats.ic3_active,
            "IC4_age": stats.ic4_age,
            "IC5_branch": stats.ic5_branch,
            "IC6_workflows": stats.ic6_workflows,
            "EC3_keywords": stats.ec3_keywords,
            "EC5_no_history": stats.ec5_no_history,  # subsumed by IC6 (always 0)
            "EC6_docs_files": stats.ec6_docs_files,
            "EC7_invalid_path": stats.ec7_invalid_path,
        },
        "reduction_pct": (stats.total_repos - stats.final_repos) / stats.total_repos * 100
        if stats.total_repos > 0
        else 0,
    }
    from dataset_utils.hashing import compute_file_hash
    report["input_sha256"] = input_sha
    report["output_file"] = str(output_path)
    report["output_sha256"] = compute_file_hash(output_path)
    with report_path.open("w") as rf:
        json.dump(report, rf, indent=2)

    return stats

def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Apply offline inclusion/exclusion criteria to the dataset."
    )
    parser.add_argument(
        "--input",
        type=Path,
        default=Path("data/6_dataset.csv"),
        help="Input dataset CSV file path.",
    )
    parser.add_argument(
        "--output",
        type=Path,
        default=Path("data/7_dataset_filtered.csv"),
        help="Output filtered dataset CSV file path.",
    )
    return parser.parse_args()

def main() -> None:
    args = parse_args()
    stats = apply_offline_criteria(args.input, args.output)

    print("Offline criteria application finished.")
    print(f"Input: {args.input}")
    print(f"Output: {args.output}")
    print(f"Total rows: {stats.total_rows}")
    print(f"Total repos: {stats.total_repos}")
    print(f"Final rows: {stats.final_rows}")
    print(f"Final repos: {stats.final_repos}")
    print(f"Reduction: {(stats.total_repos - stats.final_repos) / stats.total_repos * 100:.1f}%")
    print()
    print("Excluded by criterion:")
    print(f"  IC1 (stars < 100): {stats.ic1_stars}")
    print(f"  IC2 (commits < 100): {stats.ic2_commits}")
    print(f"  IC3 (inactive > 12mo): {stats.ic3_active}")
    print(f"  IC4 (age < 6mo): {stats.ic4_age}")
    print(f"  IC5 (not main/master): {stats.ic5_branch}")
    print(f"  IC6 (< 2 workflows): {stats.ic6_workflows}")
    print(f"  EC3 (keywords in name): {stats.ec3_keywords}")
    print(f"  EC5 (no modification history): {stats.ec5_no_history}")
    print(f"  EC6 (docs/disabled keywords):  {stats.ec6_docs_files}")
    print(f"  EC7 (invalid path structure):  {stats.ec7_invalid_path}")

main()

Step 7 — starting: apply offline IC/EC criteria...


  reading input CSV (may take a moment for large files)...
  writing output...
Offline criteria application finished.
Input: data/6_dataset.csv
Output: data/7_dataset_filtered.csv
Total rows: 253921
Total repos: 48982
Final rows: 179048
Final repos: 28321
Reduction: 42.2%

Excluded by criterion:
  IC1 (stars < 100): 0
  IC2 (commits < 100): 0
  IC3 (inactive > 12mo): 8350
  IC4 (age < 6mo): 0
  IC5 (not main/master): 4849
  IC6 (< 2 workflows): 11459
  EC3 (keywords in name): 329
  EC5 (no modification history): 0
  EC6 (docs/disabled keywords):  6748
  EC7 (invalid path structure):  896


---

**Next step:** [`08_online_enrich_dataset.ipynb`](./08_online_enrich_dataset.ipynb)
